In [ ]:
import random
import pickle
import os

# =============================================================================
# 1. KESİN FASTA OKUMA MOTORU (DOĞRUDAN ADIYLA)
# =============================================================================
hedef_klasor = "data_and_cache"
# Tam dosya adını sisteme doğrudan dayatıyoruz, arama yapıp şaşmasına izin vermiyoruz:
fasta_dosya_yolu = os.path.join(hedef_klasor, "mtor_referans_31.fasta")

print(f"🎯 Hedef Biyolojik Dosya: {fasta_dosya_yolu}\n")

def kesin_fasta_oku(dosya_yolu):
    gen_sozlugu = {}
    mevcut_gen_adi = None
    mevcut_dizi = []
    
    if not os.path.exists(dosya_yolu):
        raise FileNotFoundError(
            f"❌ HATA: Klasörde '{os.path.basename(dosya_yolu)}' adında bir dosya bulunamadı!\n"
            f"Lütfen 'Jupyter Cognac' klasöründeki o dosyanın adının tam olarak "
            f"'tgf_beta.fasta' olduğundan emin olun."
        )
        
    with open(dosya_yolu, "r", encoding="utf-8", errors="ignore") as f:
        for satir in f:
            satir_temiz = satir.strip()
            if not satir_temiz:
                continue
            
            # Satırda '>' varsa bu yeni bir gen başlığıdır
            if ">" in satir_temiz:
                if mevcut_gen_adi and mevcut_dizi:
                    gen_sozlugu[mevcut_gen_adi] = "".join(mevcut_dizi).upper()
                
                raw_baslik = satir_temiz.split(">")[-1].strip()
                mevcut_gen_adi = raw_baslik.split()[0] if raw_baslik else f"Gen_{len(gen_sozlugu)+1}"
                mevcut_dizi = []
            else:
                # Sadece DNA harflerini temizleyerek ekle
                temiz_dna = "".join([c for c in satir_temiz if c.isalpha()])
                if temiz_dna:
                    mevcut_dizi.append(temiz_dna)
                    
        if mevcut_gen_adi and mevcut_dizi:
            gen_sozlugu[mevcut_gen_adi] = "".join(mevcut_dizi).upper()
            
    return gen_sozlugu

# Okuyucuyu çalıştır
gen_havuzu_sozluk = kesin_fasta_oku(fasta_dosya_yolu)
gen_isimleri = list(gen_havuzu_sozluk.keys())
tum_gen_dizilimleri = list(gen_havuzu_sozluk.values())

print("=================================================")
print(f"🧬 mTOR FASTA ANALİZ RAPORU")
print("=================================================")
print(f"✅ BAŞARILI! Orijinal dosyadan toplam {len(gen_havuzu_sozluk)} adet gerçek gen başarıyla ayrıştırıldı.\n")

if len(gen_havuzu_sozluk) > 0:
    for i in range(min(5, len(gen_isimleri))):
        print(f" 🧬 {i+1}. Gen Adı: {gen_isimleri[i]:<15} | Orijinal Uzunluk: {len(tum_gen_dizilimleri[i])} baz")
print("=================================================\n")

if len(gen_havuzu_sozluk) == 0:
    print("⚠️ HATA: Gen sayısı hala 0! Dosya boş olabilir mi ya da uzantısı farklı mı?")
    print("Lütfen durun ve popülasyon üretimine geçmeyin.")


# =============================================================================
# 2. HAPLOİD HAVUZ ÜRETİM MOTORU (YALNIZCA GEN VARSA ÇALIŞIR)
# =============================================================================
def haploid_havuz_olustur(gen_dizileri, sayi, grup_tipi="kontrol"):
    havuz = []
    for i in range(sayi):
        birey_genleri = []
        for dna in gen_dizileri:
            polymorfik = [random.randint(100*j, 100*(j+1)-1) for j in range(len(dna)//100)]
            patojenik = [random.randint(200*j, 200*(j+1)-1) for j in range(len(dna)//200)]
            dna_liste = list(dna)
            
            for deg in random.choices(polymorfik, k=(len(polymorfik)*2)//5):
                if deg < len(dna_liste):
                    dna_liste[deg] = random.choice(list({'A','T','C','G'}.difference({dna_liste[deg]})))
            
            oran = len(patojenik)//4 if grup_tipi == "kontrol" else (len(patojenik)*3)//10
            for deg in random.choices(patojenik, k=oran):
                if deg < len(dna_liste):
                    dna_liste[deg] = random.choice(list({'A','T','C','G'}.difference({dna_liste[deg]})))
                    
            birey_genleri.append("".join(dna_liste))
        havuz.append(birey_genleri)
    return havuz

if len(gen_havuzu_sozluk) > 0:
    print("🧬 Haploid alel havuzları oluşturuluyor (Bu işlem biraz sürebilir)...")
    kontrol_haploid_havuzu = haploid_havuz_olustur(tum_gen_dizilimleri, 800, grup_tipi="kontrol")
    hasta_haploid_havuzu = haploid_havuz_olustur(tum_gen_dizilimleri, 800, grup_tipi="hasta")

    haploid_dataset = {
        "controls_haploid": kontrol_haploid_havuzu,
        "patients_haploid": hasta_haploid_havuzu,
        "gene_names": gen_isimleri  
    }

    haploid_save_path = os.path.join(hedef_klasor, "mTOR_HAPLOID_pool.pkl")
    with open(haploid_save_path, "wb") as f:
        pickle.dump(haploid_dataset, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"\n**************************************************")
    print(f"✅ 1. AŞAMA TAMAM! Gerçek 31 Gen İçin Haploid Havuz Mühürlendi.")
    print(f"Havuzdaki Kontrol Haploid Sayısı: {len(kontrol_haploid_havuzu)}")
    print(f"Havuzdaki Hasta Haploid Sayısı: {len(hasta_haploid_havuzu)}")
    print(f"Dosya: tgf_beta_HAPLOID_pool.pkl")
    print(f"**************************************************")

🎯 Hedef Biyolojik Dosya: data_and_cache\mtor_referans_31.fasta

🧬 mTOR FASTA ANALİZ RAPORU
✅ BAŞARILI! Orijinal dosyadan toplam 31 adet gerçek gen başarıyla ayrıştırıldı.

 🧬 1. Gen Adı: RICTOR          | Orijinal Uzunluk: 136480 baz
 🧬 2. Gen Adı: FKBP1A          | Orijinal Uzunluk: 24077 baz
 🧬 3. Gen Adı: MLST8           | Orijinal Uzunluk: 4000 baz
 🧬 4. Gen Adı: RHEB            | Orijinal Uzunluk: 53884 baz
 🧬 5. Gen Adı: LAMTOR1         | Orijinal Uzunluk: 6006 baz

🧬 Haploid alel havuzları oluşturuluyor (Bu işlem biraz sürebilir)...

**************************************************
✅ 1. AŞAMA TAMAM! Gerçek 31 Gen İçin Haploid Havuz Mühürlendi.
Havuzdaki Kontrol Haploid Sayısı: 800
Havuzdaki Hasta Haploid Sayısı: 800
Dosya: tgf_beta_HAPLOID_pool.pkl
**************************************************


In [10]:
import random
import pickle
import os

hedef_klasor = "data_and_cache"
haploid_dosya_yolu = os.path.join(hedef_klasor, "mTOR_HAPLOID_pool.pkl")

# 1. Haploid havuzu geri yüklüyoruz
with open(haploid_dosya_yolu, "rb") as f:
    haploid_data = pickle.load(f)

kontrol_havuzu = haploid_data["controls_haploid"]
hasta_havuzu = haploid_data["patients_haploid"]
gen_isimleri = haploid_data["gene_names"]

# 2. IUPAC Dönüşüm Kuralları
iupac_sozlugu = {
    ('A', 'G'): 'R', ('G', 'A'): 'R',
    ('C', 'T'): 'Y', ('T', 'C'): 'Y',
    ('A', 'C'): 'M', ('C', 'A'): 'M',
    ('G', 'T'): 'K', ('T', 'G'): 'K',
    ('C', 'G'): 'S', ('G', 'C'): 'S',
    ('A', 'T'): 'W', ('T', 'A'): 'W'
}

def iupac_diploid_birlestir(alell1_birey, alell2_birey):
    iupac_birey_genleri = []
    for g in range(len(alell1_birey)):
        dizi1 = alell1_birey[g]
        dizi2 = alell2_birey[g]
        iupac_gen_dizisi = []
        
        for h1, h2 in zip(dizi1, dizi2):
            if h1 == h2:
                iupac_gen_dizisi.append(h1)
            else:
                kod = iupac_sozlugu.get((h1, h2), 'N')
                iupac_gen_dizisi.append(kod)
        iupac_birey_genleri.append("".join(iupac_gen_dizisi))
    return iupac_birey_genleri

def populyasyon_diploidlestir_iupac(haploid_havuz, hedef_birey_sayisi):
    diploid_populasyon = []
    random.shuffle(haploid_havuz)
    for i in range(hedef_birey_sayisi):
        alell_1 = haploid_havuz[2*i]
        alell_2 = haploid_havuz[2*i + 1]
        diploid_birey = iupac_diploid_birlestir(alell_1, alell_2)
        diploid_populasyon.append(diploid_birey)
    return diploid_populasyon

print("🎲 Havuzdan rastgele seçim yapılıyor ve IUPAC ile Diploidleştiriliyor...")
diploid_kontroller = populyasyon_diploidlestir_iupac(kontrol_havuzu, 400)
diploid_hastalar = populyasyon_diploidlestir_iupac(hasta_havuzu, 400)

# Nihai Veri Setini Paketleme
final_dataset = {
    "controls": diploid_kontroller,
    "patients": diploid_hastalar,
    "gene_names": gen_isimleri  
}

pickle_save_path = os.path.join(hedef_klasor, "mtor_data.pkl")
with open(pickle_save_path, "wb") as f:
    pickle.dump(final_dataset, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"\n**************************************************")
print(f"✅ 2. AŞAMA TAMAM! Popülasyon IUPAC kodlarıyla diploidleştirildi.")
print(f"Nihai Veri Seti Kaydedildi: mTOR_data.pkl")
print(f"**************************************************")

🎲 Havuzdan rastgele seçim yapılıyor ve IUPAC ile Diploidleştiriliyor...

**************************************************
✅ 2. AŞAMA TAMAM! Popülasyon IUPAC kodlarıyla diploidleştirildi.
Nihai Veri Seti Kaydedildi: mTOR_data.pkl
**************************************************


In [11]:
import pickle
import os
from collections import Counter

# Veri yolu tanımlaması
hedef_klasor = "data_and_cache"
pickle_dosya_yolu = os.path.join(hedef_klasor, "mTOR_data.pkl")

print("🔍 IUPAC DİPLOİDLEŞTİRME VE VARYASYON ANALİZİ BAŞLATILDI...")
print(f"📂 İncelenen Dosya: {pickle_dosya_yolu}\n")

if not os.path.exists(pickle_dosya_yolu):
    raise FileNotFoundError("❌ Analiz edilecek nihai .pkl dosyası bulunamadı! Lütfen önce diploidleştirme hücresini çalıştırın.")

# Veriyi yükle
with open(pickle_dosya_yolu, "rb") as f:
    data = pickle.load(f)

kontroller = data.get("controls", [])
hastalar = data.get("patients", [])
gen_isimleri = data.get("gene_names", [])

# IUPAC heterozigot harfleri listesi
iupac_harfleri = {'R', 'Y', 'M', 'K', 'S', 'W'}

# Genel istatistik tutucular
kontrol_iupac_sayisi = 0
hasta_iupac_sayisi = 0
kontrol_toplam_baz = 0
hasta_toplam_baz = 0

print("======================================================================")
print("🧬 GRUP BAZLI IUPAC HETEROZİGOT DAĞILIM RAPORU")
print("======================================================================")

# Kontrol Grubu Analizi
for birey in kontroller:
    for gen_dizisi in birey:
        kontrol_toplam_baz += len(gen_dizisi)
        kontrol_iupac_sayisi += sum(1 for harf in gen_dizisi if harf in iupac_harfleri)

# Hasta Grubu Analizi
for birey in hastalar:
    for gen_dizisi in birey:
        hasta_toplam_baz += len(gen_dizisi)
        hasta_iupac_sayisi += sum(1 for harf in gen_dizisi if harf in iupac_harfleri)

kontrol_oran = (kontrol_iupac_sayisi / kontrol_toplam_baz) * 100 if kontrol_toplam_baz else 0
hasta_oran = (hasta_iupac_sayisi / hasta_toplam_baz) * 100 if hasta_toplam_baz else 0

print(f"🟢 KONTROL GRUBU:")
print(f"   - Toplam Üretilen Baz Sayısı  : {kontrol_toplam_baz:,} baz")
print(f"   - Saptanan IUPAC Heterozigot  : {kontrol_iupac_sayisi:,} adet")
print(f"   - Genomik Heterozigot Oranı   : %{kontrol_oran:.4f}")
print(f"\n🔴 HASTA GRUBU:")
print(f"   - Toplam Üretilen Baz Sayısı  : {hasta_toplam_baz:,} baz")
print(f"   - Saptanan IUPAC Heterozigot  : {hasta_iupac_sayisi:,} adet")
print(f"   - Genomik Heterozigot Oranı   : %{hasta_oran:.4f}")
print("======================================================================\n")


print("======================================================================")
print("📊 İLK 10 GEN İÇİN DETAYLI IUPAC ORAN ANALİZİ")
print("======================================================================")
print(f"{'No':<5} | {'Gen Adı':<12} | {'Kontrol IUPAC %':<18} | {'Hasta IUPAC %':<15} | {'Biyolojik Durum'}")
print("-" * 75)

for i in range(min(10, len(gen_isimleri))):
    gen_adi = gen_isimleri[i]
    
    # Kontrollerde bu genin durumu
    k_gen_baz = sum(len(birey[i]) for birey in kontroller)
    k_gen_iupac = sum(sum(1 for h in birey[i] if h in iupac_harfleri) for birey in kontroller)
    k_gen_yuzde = (k_gen_iupac / k_gen_baz) * 100 if k_gen_baz else 0
    
    # Hastalarda bu genin durumu
    h_gen_baz = sum(len(birey[i]) for birey in hastalar)
    h_gen_iupac = sum(sum(1 for h in birey[i] if h in iupac_harfleri) for birey in hastalar)
    h_gen_yuzde = (h_gen_iupac / h_gen_baz) * 100 if h_gen_baz else 0
    
    # Kontrol mekanizması doğrulaması
    durum = "✅ mTor Kurallarına Uygun (Hasta > Kontrol)" if h_gen_yuzde > k_gen_yuzde else "⚠️ Varyasyon Eşit/Düşük"
    
    print(f"{i+1:<5} | {gen_adi:<12} | %{k_gen_yuzde:<16.4f} | %{h_gen_yuzde:<14.4f} | {durum}")

print("======================================================================")
print("🏁 DOĞRULAMA SONUCU")
print("======================================================================")
if hasta_oran > kontrol_oran and kontrol_iupac_sayisi > 0:
    print("🎉 TEBRİKLER! IUPAC kodları pürüzsüzce eklenmiş.")
    print("Hasta popülasyonundaki heterozigot patojenik yük, kontrol grubundan matematiksel olarak daha yüksek.")
    print("Veri setiniz yapay zeka (CGR/CNN) eğitimine girmek için biyolojik olarak tamamen kusursuzdur!")
else:
    print("⚠️ Analizde beklenmeyen bir durum oluştu. Lütfen haploid veya diploidleştirme adımlarını kontrol edin.")
print("======================================================================")

🔍 IUPAC DİPLOİDLEŞTİRME VE VARYASYON ANALİZİ BAŞLATILDI...
📂 İncelenen Dosya: data_and_cache\mTOR_data.pkl

🧬 GRUP BAZLI IUPAC HETEROZİGOT DAĞILIM RAPORU
🟢 KONTROL GRUBU:
   - Toplam Üretilen Baz Sayısı  : 892,898,400 baz
   - Saptanan IUPAC Heterozigot  : 7,396,800 adet
   - Genomik Heterozigot Oranı   : %0.8284

🔴 HASTA GRUBU:
   - Toplam Üretilen Baz Sayısı  : 892,898,400 baz
   - Saptanan IUPAC Heterozigot  : 7,704,349 adet
   - Genomik Heterozigot Oranı   : %0.8628

📊 İLK 10 GEN İÇİN DETAYLI IUPAC ORAN ANALİZİ
No    | Gen Adı      | Kontrol IUPAC %    | Hasta IUPAC %   | Biyolojik Durum
---------------------------------------------------------------------------
1     | RICTOR       | %0.8284           | %0.8623         | ✅ mTor Kurallarına Uygun (Hasta > Kontrol)
2     | FKBP1A       | %0.8305           | %0.8678         | ✅ mTor Kurallarına Uygun (Hasta > Kontrol)
3     | MLST8        | %0.8473           | %0.8778         | ✅ mTor Kurallarına Uygun (Hasta > Kontrol)
4     | RHEB 